# M3 · Inside the machine 2: self-attention and two heads

**Tiny Transformer, episode M3.** After the first stage, each of our line's 32 rows holds one character and its place, and
nothing about its neighbours. To predict what follows *“and my p”*, the last row must use the characters before it.
**Attention** is the only step where rows pass information to each other. Here you run the block's first half yourself, on the
trained model and our running line, *“You taught me language; and my p”*: LayerNorm, queries, keys and values, scores and the mask, shares and the
mix, two heads, the output projection, and the add.

[Open in Google Colab](https://colab.research.google.com/github/gkiri/tiny_repo/blob/main/tiny-transformer/notebooks/M3-inside-attention.ipynb) (execution uses Google's hosted runtime; no graphics card needed). On your own computer:
`pip install -r requirements.txt`, then `jupyter notebook M3-inside-attention.ipynb`.

Every number below is computed here, from the trained model's 9,417 learned numbers (`prompter.json`, the recorded training
run at its best step). Where the course's page shows a recorded number from the same run, a cell asserts that this notebook
computed the same one, to 4 decimals.

**Predict first:** to guess the character after the final p, which earlier character would you look at first?

In [1]:
#@title Setup: run me first (one click) { display-mode: "form" }
# In Google Colab this fetches the course's labs (notebooks, data and the trained model) and moves into this notebook's
# folder, so every file is where the cells below look for it. On your own computer, inside a copy of the labs, it does nothing.
import os, subprocess, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/tiny_repo"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/gkiri/tiny_repo", "/content/tiny_repo"], check=True)
    os.chdir("/content/tiny_repo/tiny-transformer/notebooks")
print("ready:", os.path.basename(os.getcwd()))

ready: notebooks


In [2]:
import base64, json, math, os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
print(torch.__version__)

2.10.0+cpu


**The model's code.** The next cell is the model's whole file, `training/prompter/model.py`, copied in unchanged. The one
after it is this episode's teaching code, `code/inside_attention.py`, also unchanged: five small functions that do the block's
first half one step at a time, on the model's own parts. Both keep their files' line numbers, as the page numbers them.

In [3]:
"""Our tiny transformer: 9,417 learned numbers that predict the next character of Shakespeare.

It reads up to 32 characters of text and gives a probability to each of the 65 characters that could come next. Writing is
that prediction in a loop: pick a character, append it, slide the 32-character window, and predict again. (The class is
called Prompter for the course's file names; learners meet it as "our tiny transformer".)

The architecture is the "10k" configuration of maxpolaczuk/tiny-character-transformer (a 1-block, 2-head character GPT on Tiny
Shakespeare). The code here is written fresh for the course (that repository has no licence, so none of its code or weights
are copied): every line is one the course teaches.

    characters ─► token table (65 × 28) + position table (32 × 28)        lookup:      what, and where
               ─► one block: LayerNorm ─► 2-head causal attention ─► +    look back:   which earlier letters matter
                             LayerNorm ─► feed-forward 28 → 56 → 28 ─► +  think:       transform each row on its own
               ─► final LayerNorm ─► readout with the token table again   score all 65 characters
               ─► softmax                                                  probabilities for the next character

Parameters: token table 1,820 · positions 896 · block 6,580 (two LayerNorms 112, q|k|v 2,436, attention out 812,
feed-forward 1,624 + 1,596) · final LayerNorm 56 · readout bias 65 = 9,417.
"""

import math
from dataclasses import asdict, dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass(frozen=True)
class Config:
    vocab_size: int = 65    # every distinct character in Tiny Shakespeare
    block_size: int = 32    # how many characters the model can read at once (its context)
    d_model: int = 28       # how many numbers describe one character in one place (a row)
    n_head: int = 2         # attention heads, each looking with 28 / 2 = 14 numbers
    d_ff: int = 56          # the feed-forward network's hidden width (2 × d_model)
    n_layer: int = 1        # transformer blocks

    def to_dict(self):
        return asdict(self)


class Tokenizer:
    """One id per character, in sorted order: '\\n' is 0, ' ' is 1, … 'z' is 64."""

    def __init__(self, text: str):
        self.chars = sorted(set(text))
        self.stoi = {c: i for i, c in enumerate(self.chars)}

    def encode(self, s: str) -> list[int]:
        return [self.stoi[c] for c in s]

    def decode(self, ids) -> str:
        return "".join(self.chars[int(i)] for i in ids)


class CausalSelfAttention(nn.Module):
    """Each row asks a question (q), every earlier row offers a label (k) and a message (v). A row reads the messages of the
    rows whose labels match its question best, never a row to its right."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.n_head, self.head_dim = cfg.n_head, cfg.d_model // cfg.n_head
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model)     # q, k and v for both heads in one matrix
        self.proj = nn.Linear(cfg.d_model, cfg.d_model)        # mix the heads' results back into one row
        mask = torch.tril(torch.ones(cfg.block_size, cfg.block_size))
        self.register_buffer("mask", mask.view(1, 1, cfg.block_size, cfg.block_size), persistent=False)

    def forward(self, x, keep=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_head, self.head_dim).transpose(1, 2) for t in (q, k, v))
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)          # how well each question matches each label
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))   # no peeking at the future
        att = F.softmax(att, dim=-1)                                          # shares that add up to 1
        if keep is not None:
            keep["attn"] = att.detach()
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)            # each row: a weighted mix of messages
        return self.proj(y)


class Block(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ff = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_ff), nn.GELU(), nn.Linear(cfg.d_ff, cfg.d_model))

    def forward(self, x, keep=None):
        x = x + self.attn(self.ln1(x), keep)    # look back, and add what was found to the row
        x = x + self.ff(self.ln2(x))            # think about each row on its own, and add that too
        return x


class Prompter(nn.Module):
    def __init__(self, cfg: Config = Config()):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)    # one learned row per character
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model)    # one learned row per place 0 … 31
        self.blocks = nn.Sequential(*[Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head_bias = nn.Parameter(torch.zeros(cfg.vocab_size))  # the readout reuses tok_emb (tied); only its bias is new
        self.apply(self._init)

    @staticmethod
    def _init(m):
        # small random numbers (σ = 0.02), so the untrained model's guesses are nearly even: its first loss is about
        # ln 65 = 4.17, the loss of a uniform guess over 65 characters
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, keep=None):
        T = idx.size(1)
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))   # what + where
        for b in self.blocks:
            x = b(x, keep)
        x = self.ln_f(x)
        logits = x @ self.tok_emb.weight.T + self.head_bias                         # a score for every character
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, n, temperature=1.0, generator=None):
        for _ in range(n):
            logits, _ = self(idx[:, -self.cfg.block_size:])
            probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
            nxt = torch.multinomial(probs, 1, generator=generator)
            idx = torch.cat([idx, nxt], dim=1)
        return idx


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

In [4]:
"""M3 · Inside the machine 2: self-attention and two heads (ADR 0011). The first half of our tiny transformer's block, done one
step at a time on top of training/prompter/model.py's own parts:

  layer_norm(x, ln)       each row to average 0 and spread 1, then the learned scale and shift: [T×28] → [T×28]
  causal_mask(T)          which keys each query may see: [T×T], True on and below the diagonal
  one_head(model, x, h)   head h on the LayerNorm copy: q, k, v [T×14], scores and shares [T×T], the mix [T×14]
  multi_head(model, x)    both heads joined [T×28], then the output projection [T×28]
  attention_half(model, x0)   x1 = x0 + the projection: the residual add

code/test_inside_attention.py checks each one against model.py's own forward pass and the recorded run (prompter-stages.json);
notebooks/make_m3_inside_attention.py copies this file into M3-inside-attention.ipynb."""
import math

import torch


def layer_norm(x, ln):
    """LayerNorm, written out: ln is the model's nn.LayerNorm, which holds the learned scale (weight) and shift (bias)."""
    mean = x.mean(dim=-1, keepdim=True)                       # [T × 1]: each row's average
    var = x.var(dim=-1, unbiased=False, keepdim=True)         # [T × 1]: each row's spread, squared
    normed = (x - mean) / torch.sqrt(var + ln.eps)            # [T × 28]: average 0, spread 1
    return normed * ln.weight + ln.bias                       # [T × 28]: the 28 learned scales and 28 learned shifts


def causal_mask(T):
    """Query i may see keys 0 … i: itself and every earlier row, never a later one."""
    return torch.tril(torch.ones(T, T, dtype=torch.bool))     # [T × T]: True where key j ≤ query i


def one_head(model, x, h):
    """Head h of the block's attention, on the LayerNorm copy x [T × 28]."""
    attn = model.blocks[0].attn
    T, C = x.shape                                            # T rows of C = 28
    Dh = attn.head_dim                                        # 14 numbers per head
    q, k, v = attn.qkv(x).split(C, dim=1)                     # each [T × 28]: both heads side by side, from one layer
    cols = slice(h * Dh, (h + 1) * Dh)                        # head h's 14 columns
    q, k, v = q[:, cols], k[:, cols], v[:, cols]              # each [T × 14]
    scores = (q @ k.T) / math.sqrt(Dh)                        # [T × T]: query i · key j, divided by √14
    masked = scores.masked_fill(~causal_mask(T), float("-inf"))   # [T × T]: later keys get −∞
    shares = torch.softmax(masked, dim=-1)                    # [T × T]: each row adds up to 1; a later key gets 0
    mix = shares @ v                                          # [T × 14]: each row, a share-weighted mix of values
    return {"q": q, "k": k, "v": v, "scores": scores, "shares": shares, "mix": mix}


def multi_head(model, x):
    """Both heads on the LayerNorm copy x [T × 28]: their mixes joined, then the output projection."""
    attn = model.blocks[0].attn
    mixes = [one_head(model, x, h)["mix"] for h in range(attn.n_head)]   # 2 × [T × 14]
    joined = torch.cat(mixes, dim=-1)                         # [T × 28]: head 0's 14 numbers, then head 1's
    return joined, attn.proj(joined)                          # [T × 28], [T × 28]: the projection mixes the two


def attention_half(model, x0):
    """The block's first half: LayerNorm, both heads, the projection, and the add onto the stream."""
    blk = model.blocks[0]
    copy = layer_norm(x0, blk.ln1)                            # [T × 28]: only a copy is rescaled
    _, proj = multi_head(model, copy)                         # [T × 28]
    return x0 + proj                                          # [T × 28]: x1, the stream with attention's result added

**The trained numbers, and the rows from M2.** In Colab the Setup cell fetched the numbers with the labs; inside a copy of the
course they are found there. `x0` is what the first stage hands on: token rows plus position rows.

In [5]:
here = os.path.abspath('')
paths = [os.path.join(here, '..', 'model', 'prompter.json'),                                                     # in the labs
         os.path.join(here, '..', '..', '..', '..', 'src', 'specimen', 'tiny-transformer', 'data', 'prompter.json')]   # in the course
found = [p for p in paths if os.path.exists(p)]
assert found, "the model's trained numbers are not here: run the Setup cell at the top"
exp = json.load(open(found[0]))
state = {t['name']: torch.tensor(np.frombuffer(base64.b64decode(t['data']), dtype='<f4').reshape(t['shape']).copy()) for t in exp['weights']['tensors']}
cfg = Config(**exp['config'])
model = Prompter(cfg)
model.load_state_dict(state)
model.eval()
torch.set_grad_enabled(False)                  # this notebook only reads the model; nothing here trains it
tok = Tokenizer(exp['vocab'])
line = 'You taught me language; and my p'
ids = torch.tensor(tok.encode(line))
x0 = model.tok_emb(ids) + model.pos_emb(torch.arange(len(ids)))      # [32 × 28]: the rows from M2
blk = model.blocks[0]                                                # the model's one block
print('trained numbers from step', exp['checkpoint_step'], ':', count_params(model), 'learned numbers')
print('ids', tuple(ids.shape), '→ x0', tuple(x0.shape))
assert exp['checkpoint_step'] == 98000 and ids.tolist() == [37, 53, 59, 1, 58, 39, 59, 45, 46, 58, 1, 51, 43, 1, 50, 39, 52, 45, 59, 39, 45, 43, 11, 1, 39, 52, 42, 1, 51, 63, 1, 54] and tuple(x0.shape) == (32, 28)
assert torch.allclose(x0[31, :4], torch.tensor([-0.5172, 0.5992, 0.6395, 0.2567]), atol=2e-4)    # the recorded x0, row 31

trained numbers from step 98000 : 9417 learned numbers
ids (32,) → x0 (32, 28)


# Section 1 · Into the block

The block's plan, and the step that prepares each row for attention. Each chapter below follows its page: Picture it, Watch it
on our line, then Read and run the code (and Go deeper where the page has it).

## M3.1 · The block

*What does the block do to the rows?*

**Predict first** (the page asks it too, before its clip): our tiny transformer has 9,417 learned numbers. About how many of them sit inside
this one block: 10%, 40% or 70%?

### M3.1.1 · Picture it

Picture the rows as a stream flowing through the model. A step could replace the stream with its result, but then it would
have to rebuild everything the rows held. Instead each step takes a copy, works on it and pours its result back in: pour back
0, and the stream flows on unchanged. So "change nothing" is easy to learn, and training adds only what helps. (Where the
picture breaks: what is poured back can also cancel some of what is there.)

### M3.1.2 · Watch it on our line

*(The page's recorded clip, in words.)*

The **block** takes the residual stream, `[32×28]`, and returns the same shape. Inside it runs two steps, attention and then
the feed-forward network; each first takes a LayerNorm copy of the rows and adds its result back onto the stream, so the rows
are never thrown away. It holds 6,580 learned numbers, about 70 percent of the model's 9,417. This episode follows
its first half: LayerNorm, attention, add.

### M3.1.3 · Read and run the code

The page lights `Block.forward` in `model.py` (copied in above): its first line adds attention's result, its second the
feed-forward network's. Run the cell: the block's learned numbers, part by part, and its shape in and out.

In [6]:
count = lambda m: sum(p.numel() for p in m.parameters())
parts = {'LayerNorm 1': blk.ln1, 'q|k|v layer': blk.attn.qkv, 'output projection': blk.attn.proj,
         'LayerNorm 2': blk.ln2, 'feed-forward up': blk.ff[0], 'feed-forward down': blk.ff[2]}
for name, m in parts.items():
    print(f"  {name:18s} {count(m):5,}")
print(f"  {'the block':18s} {count(blk):5,}  ({100 * count(blk) / count_params(model):.0f}% of {count_params(model):,})")
x2 = blk(x0[None])[0]
print('the block: in', tuple(x0.shape), '→ out', tuple(x2.shape))
assert [count(m) for m in parts.values()] == [56, 2436, 812, 56, 1624, 1596]
assert count(blk) == sum(count(m) for m in parts.values()) == 6580 and round(100 * count(blk) / count_params(model)) == 70
assert tuple(x2.shape) == (32, 28)

  LayerNorm 1           56
  q|k|v layer        2,436
  output projection    812
  LayerNorm 2           56
  feed-forward up    1,624
  feed-forward down  1,596
  the block          6,580  (70% of 9,417)
the block: in (32, 28) → out (32, 28)


### ✋ Your turn

How many of the block's learned numbers belong to its first half, the one this episode follows: LayerNorm 1, the q|k|v layer and the output projection? Put the total in `answer`.

In [7]:
answer = None   # TODO: add count(...) of the three parts

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == count(blk.ln1) + count(blk.attn), 'count(blk.ln1) + count(blk.attn.qkv) + count(blk.attn.proj)'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [8]:
#@title Show a solution { display-mode: "form" }
answer = count(blk.ln1) + count(blk.attn.qkv) + count(blk.attn.proj)
print('the second half holds', count(blk) - answer)

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == count(blk.ln1) + count(blk.attn), 'count(blk.ln1) + count(blk.attn.qkv) + count(blk.attn.proj)'
    print("✓ right:", answer)

the second half holds 3276
✓ right: 3304


## M3.2 · LayerNorm

*Why rescale each row first?*

**Predict first** (the page asks it too, before its clip): LayerNorm sets a copy of row 31 to average 0, then applies its learned scale and
shift. What is the copy's average after that?

### M3.2.1 · Picture it

Rows can arrive at very different sizes: one loud, with large numbers, another quiet. Attention multiplies rows together, so a
loud row would drown out the quiet ones, whatever they hold. So every row that attention reads is first brought to the same
level. Like a volume knob, this keeps each row's pattern and changes only its size; unlike one, it also moves each row's
middle to 0.

### M3.2.2 · Watch it on our line

*(The page's recorded clip, in words.)*

**LayerNorm** measures a row's average and spread, then shifts and scales a copy to average 0 and spread 1. Row 31, the p,
arrives with average 0.05 and spread 0.48; normalised, it has 0 and 1. Then
56 learned numbers, a scale and a shift for each of the 28, move it to average −0.12 and spread
0.85. Whatever size the rows grow to, attention reads rows of a steady size, and only the copy
changes: the stream itself is untouched.

### M3.2.3 · Read and run the code

The page lights `layer_norm` in `code/inside_attention.py` (copied in above): each row's average and spread, the row
normalised, then the learned scale and shift. Run the cell on row 31, and compare with the model's own LayerNorm.

In [9]:
ln1 = blk.ln1
row = x0[31]                                                                    # [28]: the p
normed = (row - row.mean()) / torch.sqrt(row.var(unbiased=False) + ln1.eps)     # [28]: average 0, spread 1
copy = layer_norm(x0, ln1)                                                      # [32 × 28]: every row, scaled and shifted
stat = lambda r: (round(r.mean().item(), 2) + 0.0, round(r.std(unbiased=False).item(), 2))
print('row 31 as it arrives : average %5.2f, spread %.2f' % stat(row), [round(v, 3) for v in row[:5].tolist()], '…')
print('normalised           : average %5.2f, spread %.2f' % stat(normed), [round(v, 3) for v in normed[:5].tolist()], '…')
print('scaled and shifted   : average %5.2f, spread %.2f' % stat(copy[31]), [round(v, 3) for v in copy[31, :5].tolist()], '…')
print('the learned scale and shift:', ln1.weight.numel(), '+', ln1.bias.numel(), '=', count(ln1), 'numbers')
print('the copy', tuple(copy.shape), '; the stream x0 unchanged:', torch.equal(x0, model.tok_emb(ids) + model.pos_emb(torch.arange(32))))
assert stat(row) == (0.05, 0.48) and stat(normed) == (0.0, 1.0)
assert stat(copy[31]) == (-0.12, 0.85) and count(ln1) == 56
assert torch.allclose(copy, ln1(x0), atol=1e-6)                                 # the same as the model's own LayerNorm
# the recorded run (the course's export): row 31 before and after, to 4 decimals
assert abs(row.mean() - 0.0534) < 2e-4 and abs(row.std(unbiased=False) - 0.4809) < 2e-4
assert abs(copy[31].mean() - -0.1246) < 2e-4 and abs(copy[31].std(unbiased=False) - 0.847) < 2e-4
assert torch.allclose(copy[31, :4], torch.tensor([-0.4733, 0.5742, 0.3475, -0.0100]), atol=2e-4)

row 31 as it arrives : average  0.05, spread 0.48 [-0.517, 0.599, 0.64, 0.257, 0.339] …
normalised           : average  0.00, spread 1.00 [-1.186, 1.135, 1.219, 0.423, 0.594] …
scaled and shifted   : average -0.12, spread 0.85 [-0.473, 0.574, 0.348, -0.01, 0.667] …
the learned scale and shift: 28 + 28 = 56 numbers
the copy (32, 28) ; the stream x0 unchanged: True


### ✋ Your turn

Normalise row 5 (the first a) yourself, without the learned scale and shift: subtract its average, then divide by its spread. Put the new spread, rounded to 2 places, in `answer`.

In [10]:
r = x0[5]
answer = None   # TODO: round(((r - r.mean()) / r.std(unbiased=False)).std(unbiased=False).item(), 2)

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == 1.0, 'subtract r.mean(), divide by r.std(unbiased=False), then measure the spread again'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [11]:
#@title Show a solution { display-mode: "form" }
r = x0[5]
n = (r - r.mean()) / r.std(unbiased=False)
answer = round(n.std(unbiased=False).item(), 2)
print('average', round(n.mean().item(), 4) + 0.0, 'spread', answer)

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == 1.0, 'subtract r.mean(), divide by r.std(unbiased=False), then measure the spread again'
    print("✓ right:", answer)

average 0.0 spread 1.0
✓ right: 1.0


# Section 2 · Look back and share

How each row looks back at the rows before it: queries, keys and scores; shares and the mix; two heads.

## M3.3 · Queries, keys and scores

*How does a row decide where to look?*

**Predict first** (the page asks it too, before its clip): one head scores every row's query against every row's key. For 32 rows, how many
scores is that, the hidden ones included?

### M3.3.1 · Picture it

The p must find which earlier rows matter, and no fixed rule says which. So every row makes 3 lists of numbers: a **query**,
the question it asks; a **key**, its label; and a **value**, its message. Draw each as an arrow: a query and a key pointing the
same way score high; at right angles, 0; opposite, below 0; longer arrows score more. The row takes more from its best
matches. (Where the picture breaks: these are learned numbers, not words.)

### M3.3.2 · Watch it on our line

*(The page's recorded clip, in words.)*

One learned layer turns each row of the LayerNorm copy into three vectors: a **query**, what the row looks for; a **key**, what
it offers; and a **value**, what it hands over. Per head each is 14 numbers, so q, k and v are `[32×14]` for each of the
2 heads, and the layer holds 2,436 learned numbers. Each query is multiplied with every key, number by number, and summed
(a dot product), then divided by √14 as `model.py` does: a grid of scores, `[32×32]` per head. The **causal mask** hides
every later row: row 31 may score all 32 rows, row 0 only itself.

### M3.3.3 · Go deeper: scale and mask

Two fixed rules in the score grid, neither learned. Each score adds up 14 products. For random numbers, as when training
starts, such a sum spreads about √14 times wider than one product, so the model divides by √14, about 3.74.
Leave it out, and row 31's largest share in head 0 becomes 1.00, not 0.88: shares pushed to 0 or 1
barely move when training nudges a score. The **mask** sets every later score to −∞, a share of exactly 0: in training all 32
rows guess at once, and none may see its answer.

In [12]:
g = torch.Generator().manual_seed(0)                                     # random numbers, as when training starts
a, b = torch.randn(100_000, 14, generator=g), torch.randn(100_000, 14, generator=g)
one, total = (a[:, 0] * b[:, 0]).std().item(), (a * b).sum(-1).std().item()
print(f"spread of one product {one:.2f}; of a sum of 14 {total:.2f}: {total / one:.2f} times wider; √14 = {math.sqrt(14):.2f}")
H = one_head(model, layer_norm(x0, blk.ln1), 0)
raw = H['q'] @ H['k'].T                                                  # the scores before the division
row = causal_mask(32)[31]
with_div = torch.softmax(H['scores'][31].masked_fill(~row, float('-inf')), -1).max().item()
without = torch.softmax(raw[31].masked_fill(~row, float('-inf')), -1).max().item()
print(f"row 31, head 0, largest share: {with_div:.2f} with ÷ √14, {without:.2f} without it")
assert abs(total / one - math.sqrt(14)) < 0.1 and abs(with_div - 0.8782) < 2e-4 and without > 0.995

spread of one product 0.99; of a sum of 14 3.73: 3.77 times wider; √14 = 3.74
row 31, head 0, largest share: 0.88 with ÷ √14, 1.00 without it


### M3.3.4 · Read and run the code

The page lights `one_head` in `code/inside_attention.py`: q, k and v from one learned layer, head h's 14 columns of each,
`q @ k.T / √14`, then `masked_fill` for every later key. Run the cell: the shapes, the mask, and the recorded numbers checked.

In [13]:
qkv = blk.attn.qkv
print('the q|k|v layer:', tuple(qkv.weight.shape), 'weights +', tuple(qkv.bias.shape), 'biases =', count(qkv), 'learned numbers')
H0 = one_head(model, copy, 0)
for name in ('q', 'k', 'v', 'scores'):
    print(f"head 0 {name:6s}", tuple(H0[name].shape))
assert count(qkv) == 2436 and tuple(H0['q'].shape) == (32, 14) and tuple(H0['scores'].shape) == (32, 32)
assert torch.allclose(H0['scores'], H0['q'] @ H0['k'].T / math.sqrt(14))       # query · key, divided by √14

mask = causal_mask(32)
print('row 31 may see', int(mask[31].sum()), 'rows; row 0 sees', int(mask[0].sum()))
print('head 0, rows and keys 0 to 5, with the mask (-inf: a later row, hidden):')
print(H0['scores'].masked_fill(~mask, float('-inf'))[:6, :6].numpy().round(2))
assert int(mask[31].sum()) == 32 and int(mask[0].sum()) == 1
assert torch.equal(mask, blk.attn.mask[0, 0, :32, :32] == 1)                    # the model's own mask
# the recorded run (the course's export): head 0's query of row 31, key and value of row 30, row 31's last 4 scores
assert torch.allclose(H0['q'][31, :4], torch.tensor([-0.9162, -0.1556, 1.2178, 5.3362]), atol=2e-4)
assert torch.allclose(H0['k'][30, :4], torch.tensor([-3.9904, -3.4468, -0.4884, 4.6312]), atol=2e-4)
assert torch.allclose(H0['v'][30, :4], torch.tensor([-0.4871, -0.1412, 1.1397, 0.0882]), atol=2e-4)
assert torch.allclose(H0['scores'][31, 28:], torch.tensor([21.8130, 23.1813, 26.0642, 23.2109]), atol=2e-4)

the q|k|v layer: (84, 28) weights + (84,) biases = 2436 learned numbers
head 0 q      (32, 14)
head 0 k      (32, 14)
head 0 v      (32, 14)
head 0 scores (32, 32)
row 31 may see 32 rows; row 0 sees 1
head 0, rows and keys 0 to 5, with the mask (-inf: a later row, hidden):
[[ -8.78   -inf   -inf   -inf   -inf   -inf]
 [-19.23 -18.72   -inf   -inf   -inf   -inf]
 [-14.   -14.69 -15.12   -inf   -inf   -inf]
 [ -6.57  -8.34  -6.71  -6.44   -inf   -inf]
 [-20.83 -20.94 -20.15 -14.8  -19.46   -inf]
 [-14.04 -14.16 -11.82 -10.12 -10.54 -10.78]]


### ✋ Your turn

How many scores per head survive the mask, over the whole `[32×32]` grid? Put the number in `answer`.

In [14]:
answer = None   # TODO: count the True entries of causal_mask(32)

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == int(causal_mask(32).sum()), 'int(causal_mask(32).sum()): row i keeps i + 1 scores'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [15]:
#@title Show a solution { display-mode: "form" }
answer = int(causal_mask(32).sum())
print(answer, 'of', 32 * 32, '=', sum(i + 1 for i in range(32)))

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == int(causal_mask(32).sum()), 'int(causal_mask(32).sum()): row i keeps i + 1 scores'
    print("✓ right:", answer)

528 of 1024 = 528
✓ right: 528


## M3.4 · Shares and the mix

*How does a row gather what it found?*

**Predict first** (the page asks it too, before its clip): the one from the top. To guess the character after the final p, which earlier
character would you look at first?

### M3.4.1 · Picture it

Every earlier row now has a score. The row could take only the best match, but then a tiny change in one score could flip the
choice: a jump. Instead the scores become shares that add up to 1, and the row blends the messages by those shares: nudge a
score, and the blend moves a little. Small, smooth changes are what training can work with. (Where the picture breaks: the
messages are lists of numbers, blended number by number, not sentences.)

### M3.4.2 · Watch it on our line

*(The page's recorded clip, in words.)*

**Softmax** turns each row's scores into shares: each between 0 and 1, all adding up to 1, and a hidden later row gets
exactly 0. In head 0, row 31, the p, gives 0.88 of its share to position 30, the space just before it: one
answer to the question at the top. Each row's result is a mix of the value vectors, weighted by its shares: `[32×14]`. Rows
reading other rows of the same text is **self-attention**; the shares are made fresh for every input, and only the layer that
makes queries, keys and values is learned.

### M3.4.3 · Read and run the code

The page lights the last 2 lines of `one_head`: `torch.softmax` turns each row of masked scores into shares, and
`shares @ v` mixes the values. Run the cells: row 31's share of every row, then change the space at position 30 and watch
row 31's mix move.

In [16]:
sh = H0['shares']
print('shares', tuple(sh.shape), '; every row adds up to 1:', torch.allclose(sh.sum(-1), torch.ones(32)),
      '; the largest share of a hidden row:', sh[~mask].max().item())
show = lambda c: {'\n': '↵', ' ': '␣'}.get(c, c)
print('head 0, row 31 (the p): its share of each row of the line')
for j in range(32):
    print(f"  {j:2d} {show(line[j])} {sh[31, j].item():.4f} {'█' * round(50 * sh[31, j].item())}")
mix = H0['mix']
print('the mix', tuple(mix.shape), '; row 31 starts', [round(v, 4) for v in mix[31, :4].tolist()], '…')
assert torch.allclose(sh.sum(-1), torch.ones(32)) and sh[~mask].max().item() == 0 and tuple(mix.shape) == (32, 14)
assert torch.allclose(mix[31], (sh[31, :, None] * H0['v']).sum(0), atol=1e-6)    # the shares' weighted sum of the values
keep = {}
model(ids[None], keep=keep)
assert torch.allclose(keep['attn'][0, 0], sh, atol=1e-6)                        # the model's own shares, head 0
# the recorded run (the course's export): row 31's three largest shares, its last 4 shares, its mix
top = sh[31].topk(3)
assert top.indices.tolist() == [30, 31, 29]
assert torch.allclose(top.values, torch.tensor([0.8782, 0.0506, 0.0492]), atol=2e-4)
assert torch.allclose(sh[31, 28:], torch.tensor([0.0125, 0.0492, 0.8782, 0.0506]), atol=2e-4)
assert torch.allclose(mix[31, :4], torch.tensor([-0.2911, -0.0971, 1.0864, 0.0485]), atol=2e-4)

shares (32, 32) ; every row adds up to 1: True ; the largest share of a hidden row: 0.0
head 0, row 31 (the p): its share of each row of the line
   0 Y 0.0000 
   1 o 0.0000 
   2 u 0.0000 
   3 ␣ 0.0000 
   4 t 0.0000 
   5 a 0.0000 
   6 u 0.0000 
   7 g 0.0000 
   8 h 0.0000 
   9 t 0.0000 
  10 ␣ 0.0000 
  11 m 0.0000 
  12 e 0.0000 
  13 ␣ 0.0000 
  14 l 0.0000 
  15 a 0.0000 
  16 n 0.0000 
  17 g 0.0000 
  18 u 0.0000 
  19 a 0.0000 
  20 g 0.0000 
  21 e 0.0000 
  22 ; 0.0000 
  23 ␣ 0.0000 
  24 a 0.0000 
  25 n 0.0000 
  26 d 0.0000 
  27 ␣ 0.0095 
  28 m 0.0125 █
  29 y 0.0492 ██
  30 ␣ 0.8782 ████████████████████████████████████████████
  31 p 0.0506 ███
the mix (32, 14) ; row 31 starts [-0.2911, -0.0971, 1.0864, 0.0485] …


Only attention lets a row read other rows. Change one earlier character, the space at position 30, and row 31's own row in
`x0` does not move at all; its mix does.

In [17]:
edited = line[:30] + '!' + line[31:]
ids_e = torch.tensor(tok.encode(edited))
x0_e = model.tok_emb(ids_e) + model.pos_emb(torch.arange(32))
mix_e = one_head(model, layer_norm(x0_e, ln1), 0)['mix']
print(repr(edited))
print("row 31 of x0, the same:", torch.equal(x0_e[31], x0[31]))
print("row 31's mix, the same:", torch.allclose(mix_e[31], mix[31]), '; it moved by', round((mix_e[31] - mix[31]).norm().item(), 3))
assert torch.equal(x0_e[31], x0[31]) and not torch.allclose(mix_e[31], mix[31])

'You taught me language; and my!p'
row 31 of x0, the same: True
row 31's mix, the same: False ; it moved by 2.288


### ✋ Your turn

Row 12 is the e of “me”. In head 0, which position gets its largest share? Put the position in `answer`, then look up which character sits there.

In [18]:
answer = None   # TODO: the position of the largest value in sh[12]

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == int(sh[12].argmax()), 'int(sh[12].argmax())'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [19]:
#@title Show a solution { display-mode: "form" }
answer = int(sh[12].argmax())
print('position', answer, 'holds', repr(line[answer]), 'with a share of', round(sh[12, answer].item(), 4))

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == int(sh[12].argmax()), 'int(sh[12].argmax())'
    print("✓ right:", answer)

position 10 holds ' ' with a share of 0.7907
✓ right: 10


## M3.5 · Two heads

*Why two heads?*

**Predict first** (the page asks it too, before its clip): the 2 heads' results are joined back into 28 numbers, then one learned layer mixes
them. How many learned numbers does that layer hold?

### M3.5.1 · Picture it

A set of shares makes a single blend, so it follows a single pattern at a time. But a row may need 2 kinds of earlier rows at
once, and a single blend would have to split its shares between them. So attention makes 2 blends side by side, each from its
own shares: 2 heads. (Where the picture breaks: nothing tells a head what to look for. Training shapes each one, and on some
rows both look at the same place.)

### M3.5.2 · Watch it on our line

*(The page's recorded clip, in words.)*

Two heads run side by side, each with its own queries, keys and values of 14 numbers. In head 1, row 31 also gives the
space most of its share, 0.84, and the p itself 0.12; across the whole line, the two heads look
hardest at different positions in 16 of the 32 rows. Their results are joined, `[32×14]` twice → `[32×28]`, mixed by
one more learned layer, the **output projection** (812 numbers), and added onto the stream: `x1 = x0 + projection`,
`[32×28]` in and out. This is **multi-head attention**, and splitting 28 numbers into two heads of 14 adds no learned
numbers: the same layer, cut into two slices.

### M3.5.3 · Go deeper: what attention changed

Measure row 31 as an arrow of 28 numbers. It comes in 2.6 long; attention's update is 2.4 long;
the row leaves 3.6 long. Not 5: the update points almost at right angles to the row,
so it adds a new direction, not more of what the row held. That direction is built from the values of the rows it looked at;
M4 measures what it does to the model's guess.

In [20]:
x1_ = attention_half(model, x0)
upd = x1_[31] - x0[31]                                                    # attention's update to row 31
cos = (x0[31] @ upd / (x0[31].norm() * upd.norm())).item()
print(f"row 31: in {x0[31].norm():.2f} long, update {upd.norm():.2f}, out {x1_[31].norm():.2f}; the angle between row and update {math.degrees(math.acos(cos)):.0f}°")
assert abs(x0[31].norm() - 2.5602) < 2e-4 and abs(upd.norm() - 2.4321) < 2e-4 and abs(x1_[31].norm() - 3.5805) < 2e-4

row 31: in 2.56 long, update 2.43, out 3.58; the angle between row and update 88°


### M3.5.4 · Read and run the code

The page lights `multi_head` and `attention_half`: one head per slice, `torch.cat` joins the 2 mixes back to 28 numbers, the
output projection mixes them, and the add gives the new rows. Run the cell: both heads, the join, the projection and `x1`.

In [21]:
H1 = one_head(model, copy, 1)
top = H1['shares'][31].topk(3)
print('head 1, row 31:', [(show(line[j]), j, round(s, 4)) for s, j in zip(top.values.tolist(), top.indices.tolist())])
most0, most1 = H0['shares'].argmax(-1), H1['shares'].argmax(-1)       # [32]: each row's most attended position, per head
differ = (most0 != most1).nonzero().flatten().tolist()
print('rows where the two heads look hardest at different positions:', len(differ), 'of 32:', differ)
assert top.indices.tolist() == [30, 31, 29] and len(differ) == 16
assert torch.allclose(top.values, torch.tensor([0.8409, 0.1248, 0.0267]), atol=2e-4)

joined, proj = multi_head(model, copy)
x1 = x0 + proj                                                       # the residual add
print('mixes', tuple(H0['mix'].shape), '+', tuple(H1['mix'].shape), '→ joined', tuple(joined.shape),
      '→ projection', tuple(proj.shape), '→ x1', tuple(x1.shape))
print('the output projection:', tuple(blk.attn.proj.weight.shape), 'weights +', tuple(blk.attn.proj.bias.shape), 'biases =',
      count(blk.attn.proj), 'learned numbers')
assert torch.equal(joined, torch.cat([H0['mix'], H1['mix']], dim=-1)) and count(blk.attn.proj) == 812
assert torch.allclose(proj, blk.attn(blk.ln1(x0[None]))[0], atol=1e-5)    # the model's own attention, both heads at once
assert torch.allclose(x1, attention_half(model, x0)) and tuple(x1.shape) == (32, 28)
# the recorded run (the course's export): row 31 joined (each head's first 4), projected, and added
assert torch.allclose(joined[31, :4], torch.tensor([-0.2911, -0.0971, 1.0864, 0.0485]), atol=2e-4)
assert torch.allclose(joined[31, 14:18], torch.tensor([0.3649, -0.1813, 0.5187, 0.6138]), atol=2e-4)
assert torch.allclose(proj[31, :4], torch.tensor([0.9624, -0.2872, 0.5321, 0.0883]), atol=2e-4)
assert torch.allclose(x1[31, :4], torch.tensor([0.4452, 0.3120, 1.1716, 0.3450]), atol=2e-4)

for n in (1, 2, 4):                                                  # more heads, not more numbers
    print(f"  attention with {n} head(s) of {28 // n:2d}: {count(CausalSelfAttention(Config(n_head=n))):,} learned numbers")
assert len({count(CausalSelfAttention(Config(n_head=n))) for n in (1, 2, 4)}) == 1

head 1, row 31: [('␣', 30, 0.8409), ('p', 31, 0.1248), ('y', 29, 0.0267)]
rows where the two heads look hardest at different positions: 16 of 32: [1, 5, 7, 8, 9, 10, 12, 15, 16, 17, 18, 20, 25, 26, 27, 29]
mixes (32, 14) + (32, 14) → joined (32, 28) → projection (32, 28) → x1 (32, 28)
the output projection: (28, 28) weights + (28,) biases = 812 learned numbers
  attention with 1 head(s) of 28: 3,248 learned numbers
  attention with 2 head(s) of 14: 3,248 learned numbers
  attention with 4 head(s) of  7: 3,248 learned numbers


### ✋ Your turn

Which head looks further back? For each head, take every row's shares times how far back each key sits (`back`, below), add them up per row, and average over the 32 rows. Put the head (0 or 1) with the larger average in `answer`.

In [22]:
back = torch.arange(32)[:, None] - torch.arange(32)[None, :]       # [32 × 32]: query i minus key j
reach = [None, None]   # TODO: for h in 0 and 1, (one_head(model, copy, h)['shares'] * back).sum(-1).mean().item()
answer = None          # TODO: the head with the larger reach

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == int(torch.tensor([(one_head(model, copy, h)['shares'] * back).sum(-1).mean() for h in (0, 1)]).argmax()), 'compute reach for both heads, then pick the head with the larger one'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [23]:
#@title Show a solution { display-mode: "form" }
back = torch.arange(32)[:, None] - torch.arange(32)[None, :]
reach = [(one_head(model, copy, h)['shares'] * back).sum(-1).mean().item() for h in (0, 1)]
answer = reach.index(max(reach))
print('average reach, in positions back: head 0', round(reach[0], 2), '; head 1', round(reach[1], 2))

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == int(torch.tensor([(one_head(model, copy, h)['shares'] * back).sum(-1).mean() for h in (0, 1)]).argmax()), 'compute reach for both heads, then pick the head with the larger one'
    print("✓ right:", answer)

average reach, in positions back: head 0 1.28 ; head 1 1.03
✓ right: 0
